# TP9 — Tracker B11-GKF dans MLflow (combler le trou identifié en TP8)

**Constat (TP8 / `ML/model_card.ipynb`)** : le modèle XGBoost retenu, **B11-GKF**,
n'a **aucun run MLflow** — le seul run persisté (`mlruns/1`, `mlflow_tp7.db`,
expérience `TP7_maintenance_predictive`) correspond au modèle antérieur **B7,
révélé fuyant** (`feature_row_id`). Ce TP ferme ce trou : on enregistre B11-GKF dans
la **même** expérience MLflow que B7, pour permettre une comparaison directe
côté-à-côté (fuyant vs corrigé) — et on en profite pour obtenir, via
`mlflow.xgboost.log_model`, le **premier artefact persisté** de ce modèle.

> ⚠ **Ce notebook vit dans `DL/` mais s'exécute contre le projet `ML/`** — autre venv
> (`ML/.venv` : xgboost, sqlalchemy, mlflow), autre base (PostgreSQL `indusense_db`),
> autre store de tracking (`ML/mlflow_tp7.db`). Toutes les résolutions de chemin
> ci-dessous sont explicites et pointent vers `ML/`, pas vers `DL/`.

| § | Contenu |
|---|---|
| §1 | Reconstruire B11-GKF (mêmes données, mêmes hyperparamètres que TP11/model_card) |
| §2 | Logger le run dans MLflow — mêmes conventions que B7 (TP7/TP8) |
| §3 | Vérifier : recharger le modèle depuis MLflow, comparer aux métriques `model_card.ipynb` |
| §4 | Comparer B7 (fuyant) vs B11-GKF (corrigé) dans le même tracking store |


## §0 — Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    confusion_matrix, precision_score, recall_score,
)
from xgboost import XGBClassifier
from codecarbon import EmissionsTracker
import mlflow
import mlflow.xgboost

RANDOM_STATE = 42

# Ce notebook est rangé dans DL/ mais opère sur le projet ML/ — chemins explicites.
ML_ROOT = Path("../ML").resolve()
assert (ML_ROOT / "mlflow_tp7.db").exists(), f"ML_ROOT introuvable ou mal résolu : {ML_ROOT}"
print(f"ML_ROOT = {ML_ROOT}")

ARTIFACTS_DIR = ML_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"mlflow {mlflow.__version__}")


ML_ROOT = C:\Users\Aelion\py-init\ML
mlflow 3.14.0


## §1 — Reconstruire B11-GKF

Mêmes données, même requête, mêmes hyperparamètres figés que `ML/model_card.ipynb`
(eux-mêmes repris de l'étude Optuna rejouée dans `TP11.ipynb`) — aucune redécouverte,
juste la reproduction exacte du modèle déjà validé.

In [2]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username="indusense_user",
    password="ThEP@ssW0rd",
    host="localhost", port=5432, database="indusense_db",
)
engine = create_engine(url)
df = pd.read_sql(
    "SELECT * FROM gold_machine_hourly_feature ORDER BY machine_id, window_start",
    engine
)

TARGET = "label_failure_next_24h"
LEAKAGE_COLS = [
    "machine_id", "ingestion_batch_id", "window_start", "window_end", "split_set",
    "label_failure_next_6h", "label_failure_next_12h", "label_failure_next_48h",
    TARGET, "feature_row_id",
]
FEATURE_COLS = [c for c in df.columns if c not in LEAKAGE_COLS]

trainval_df = df[df["split_set"].isin(["train", "validation"])].copy()
test_df     = df[df["split_set"] == "test"].copy()

X_tv, y_tv     = trainval_df[FEATURE_COLS], trainval_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

SPW_GLOBAL = round((y_tv == 0).sum() / (y_tv == 1).sum(), 2)
N_FEATURES = len(FEATURE_COLS)

BEST_PARAMS = {
    "n_estimators":     287,
    "max_depth":        9,
    "learning_rate":    0.02887139049912187,
    "subsample":        0.8252019146348859,
    "colsample_bytree": 0.5736306798333981,
    "min_child_weight": 11,
    "reg_alpha":        0.01918528344873483,
    "reg_lambda":       0.6228756133555158,
    "scale_pos_weight": SPW_GLOBAL,
    "random_state":     RANDOM_STATE,
    "verbosity":        0,
}

print(f"Train+val : {len(X_tv):,} | Test : {len(X_test):,} | Features : {N_FEATURES}")
print(f"scale_pos_weight : {SPW_GLOBAL}")


Train+val : 112,996 | Test : 19,944 | Features : 69
scale_pos_weight : 27.12


In [3]:
tracker = EmissionsTracker(
    project_name="tp9_b11_gkf_mlflow",
    output_dir=str(ARTIFACTS_DIR),
    measure_power_secs=1,
    log_level="error",
    save_to_file=True,
)
tracker.start()

pipe_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   XGBClassifier(**BEST_PARAMS)),
])
pipe_final.fit(X_tv, y_tv)

emissions_kg = tracker.stop()
carbon_data = tracker.final_emissions_data

y_prob_test = pipe_final.predict_proba(X_test)[:, 1]
y_pred_test = pipe_final.predict(X_test)
y_prob_tv   = pipe_final.predict_proba(X_tv)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

metrics = {
    "pr_auc_train":   round(float(average_precision_score(y_tv, y_prob_tv)), 4),
    "pr_auc_test":    round(float(average_precision_score(y_test, y_prob_test)), 4),
    "roc_auc_test":   round(float(roc_auc_score(y_test, y_prob_test)), 4),
    "f1_test":        round(float(f1_score(y_test, y_pred_test, zero_division=0)), 4),
    "precision_test": round(float(precision_score(y_test, y_pred_test, zero_division=0)), 4),
    "recall_test":    round(float(recall_score(y_test, y_pred_test, zero_division=0)), 4),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    "co2_gco2eq":     round(emissions_kg * 1000, 4),
    "energy_wh":      round(carbon_data.energy_consumed * 1000, 4),
}
print(json.dumps(metrics, indent=2))
print(f"\nCohérence avec model_card.ipynb (PR-AUC test=0.8799) : "
      f"{'OK' if abs(metrics['pr_auc_test'] - 0.8799) < 1e-3 else 'ECART'}")


[codecarbon WARNING @ 14:02:53] Multiple instances of codecarbon are allowed to run at the same time.


{
  "pr_auc_train": 0.9997,
  "pr_auc_test": 0.8799,
  "roc_auc_test": 0.9949,
  "f1_test": 0.7586,
  "precision_test": 0.6681,
  "recall_test": 0.8776,
  "tp": 638,
  "tn": 18900,
  "fp": 317,
  "fn": 89,
  "co2_gco2eq": 0.0023,
  "energy_wh": 0.0415
}

Cohérence avec model_card.ipynb (PR-AUC test=0.8799) : OK


## §2 — Logger dans MLflow

Même tracking store et même expérience que B7 (`ML/mlflow_tp7.db`,
`TP7_maintenance_predictive`) — pour que B7 (fuyant) et B11-GKF (corrigé)
apparaissent côte à côte dans `mlflow ui`, comparables directement.

In [4]:
mlflow.set_tracking_uri(f"sqlite:///{(ML_ROOT / 'mlflow_tp7.db').as_posix()}")
mlflow.set_experiment("TP7_maintenance_predictive")

with mlflow.start_run(run_name="XGBoost_GroupKFold_B11") as run:
    mlflow.set_tags({
        "lineage": "B11-GKF",
        "supersedes": "XGBoost_Optuna_B7 (fuite feature_row_id, corrigée depuis TP8b)",
        "source_notebook": "TP11.ipynb (recherche HP) + model_card.ipynb (métriques) + TP9.ipynb (ce run)",
        "leakage_status": "corrigée — feature_row_id exclu de FEATURE_COLS",
    })

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("target", TARGET)
    mlflow.log_param("n_features", N_FEATURES)
    mlflow.log_param("cv_strategy", "GroupKFold(15) — une machine exclue par fold")
    mlflow.log_param("optuna_objective", "PR-AUC moyen, GroupKFold(5), 30 essais")

    mlflow.log_metrics({k: v for k, v in metrics.items() if k not in ("tp", "tn", "fp", "fn")})
    mlflow.log_metrics({"tp": metrics["tp"], "tn": metrics["tn"], "fp": metrics["fp"], "fn": metrics["fn"]})

    mlflow.xgboost.log_model(pipe_final.named_steps["model"], name="xgboost_b11_gkf")

    run_id = run.info.run_id

print(f"Run MLflow enregistré : run_id={run_id}")
print(f"UI : mlflow ui --backend-store-uri sqlite:///{(ML_ROOT / 'mlflow_tp7.db').as_posix()}")


2026/07/23 14:03:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run MLflow enregistré : run_id=fa336bc0880b48bc849a4a6b1e6f412d
UI : mlflow ui --backend-store-uri sqlite:///C:/Users/Aelion/py-init/ML/mlflow_tp7.db


## §3 — Vérifier : recharger le modèle depuis MLflow

B11-GKF a maintenant un **artefact persisté** — premier de son histoire. On le
recharge depuis le tracking store (pas depuis l'objet Python en mémoire) et on
reproduit les métriques de test pour confirmer que l'artefact loggé est bien
fonctionnel.

In [5]:
loaded_model = mlflow.xgboost.load_model(f"runs:/{run_id}/xgboost_b11_gkf")

# Le modèle loggé est le XGBClassifier nu (pas le Pipeline) : on ré-applique
# l'imputation médiane manuellement pour rester fidèle au pipeline d'origine.
imputer_check = SimpleImputer(strategy="median").fit(X_tv)
X_test_imputed = imputer_check.transform(X_test)

y_prob_reloaded = loaded_model.predict_proba(X_test_imputed)[:, 1]
pr_auc_reloaded = average_precision_score(y_test, y_prob_reloaded)

print(f"PR-AUC test (run initial)    : {metrics['pr_auc_test']:.4f}")
print(f"PR-AUC test (modèle rechargé) : {pr_auc_reloaded:.4f}")
print(f"Identique : {abs(metrics['pr_auc_test'] - round(pr_auc_reloaded, 4)) < 1e-3}")


PR-AUC test (run initial)    : 0.8799
PR-AUC test (modèle rechargé) : 0.8799
Identique : True


## §4 — B7 (fuyant) vs B11-GKF (corrigé), même tracking store

In [6]:
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("TP7_maintenance_predictive")
runs = client.search_runs(experiment.experiment_id, order_by=["start_time ASC"])

rows = []
for r in runs:
    rows.append({
        "run_name": r.data.tags.get("mlflow.runName", "?"),
        "pr_auc_test": r.data.metrics.get("pr_auc_test"),
        "roc_auc_test": r.data.metrics.get("roc_auc_test"),
        "f1_test": r.data.metrics.get("f1_test"),
        "lineage": r.data.tags.get("lineage", "—"),
        "leakage_status": r.data.tags.get("leakage_status", "—"),
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


              run_name  pr_auc_test  roc_auc_test  f1_test lineage                                  leakage_status
   Logistic_Regression          NaN           NaN      NaN       —                                               —
         Random_Forest          NaN           NaN      NaN       —                                               —
               XGBoost          NaN           NaN      NaN       —                                               —
   Logistic_Regression          NaN           NaN      NaN       —                                               —
         Random_Forest          NaN           NaN      NaN       —                                               —
               XGBoost          NaN           NaN      NaN       —                                               —
     XGBoost_Optuna_B7     0.843721      0.993762 0.773045       —                                               —
     XGBoost_Optuna_B7     0.843721      0.993762 0.773045       —              

## Synthèse

**Fait dans ce TP9 :**
- B11-GKF (modèle retenu) est désormais tracké dans **le même store MLflow** que B7
  (`ML/mlflow_tp7.db`, expérience `TP7_maintenance_predictive`) — comparaison directe
  possible via `mlflow ui`.
- Premier **artefact persisté** pour B11-GKF (`mlflow.xgboost.log_model`), rechargé
  et revérifié en §3 — comble le trou signalé dans `ML/model_card.ipynb`
  ("aucun artefact persisté").
- Tags explicites (`lineage`, `supersedes`, `leakage_status`) pour qu'un lecteur du
  tracking store comprenne immédiatement que B7 est l'ancienne version fuyante et
  B11-GKF la version corrigée retenue — sans avoir à relire l'historique des notebooks.

**Reste hors périmètre** : `ML/artifacts/model_card.md` n'a pas été régénéré — sa
section *Model Sources* ("aucun artefact persisté") est maintenant obsolète depuis ce
TP9 et devrait être mise à jour pour pointer vers `runs:/{run_id}/xgboost_b11_gkf`.